# AutoResearch on Kaggle (Offline-Ready)

基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：** Settings → Accelerator → 選 GPU T4 x2 或 P100

> Kaggle 環境通常**無法連外網**，此 notebook 已將所有必要腳本內嵌，不需要下載任何檔案。

### 前置準備（一次性）
1. 在本機 `git clone https://github.com/karpathy/autoresearch.git`
2. 到 Kaggle → **Datasets** → **New Dataset** → 上傳整個 `autoresearch/` 資料夾
3. 在此 notebook 右側 **Add data** → 加入你上傳的 dataset
4. 確認 dataset 出現在 `/kaggle/input/autoresearch/` 路徑

In [ ]:
# 確認 GPU 並偵測架構
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"\n✅ GPU: {name} (compute capability: {cap[0]}.{cap[1]})")
    if cap[0] >= 8:
        print("   Ampere+ → Flash Attention 3 原生支援")
    else:
        print("   Pre-Ampere → 將自動套用 SDPA fallback patch")
else:
    print("❌ 未偵測到 GPU！請到 Settings → Accelerator → 選 GPU")

In [ ]:
# 從 Kaggle dataset 複製 autoresearch 到工作目錄（Kaggle input 是唯讀的）
import shutil, os, glob
from pathlib import Path

WORK_DIR = Path("/kaggle/working/autoresearch")

# 嘗試多種 dataset 路徑（Kaggle dataset 名稱可能不同）
INPUT_CANDIDATES = [
    "/kaggle/input/autoresearch",
    "/kaggle/input/autoresearch-repo",
    "/kaggle/input/karpathy-autoresearch",
]
# 也搜尋所有 input 目錄中包含 train.py 的
for d in glob.glob("/kaggle/input/*/"):
    if os.path.exists(os.path.join(d, "train.py")):
        INPUT_CANDIDATES.insert(0, d.rstrip("/"))

input_dir = None
for candidate in INPUT_CANDIDATES:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, "train.py")):
        input_dir = candidate
        break

if input_dir:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(input_dir, str(WORK_DIR))
    print(f"✅ Copied autoresearch from {input_dir} → {WORK_DIR}")
else:
    # Fallback: try git clone (only works if internet is enabled)
    print("⚠️ Autoresearch dataset not found in /kaggle/input/")
    print("   Trying git clone (requires internet)...")
    ret = os.system("git clone https://github.com/karpathy/autoresearch.git /kaggle/working/autoresearch")
    if ret != 0:
        raise RuntimeError(
            "❌ Could not find autoresearch!\n"
            "   請先上傳 autoresearch 為 Kaggle Dataset：\n"
            "   1. 本機執行: git clone https://github.com/karpathy/autoresearch.git\n"
            "   2. Kaggle → Datasets → New Dataset → 上傳 autoresearch/ 資料夾\n"
            "   3. 在此 notebook 右側 Add data → 加入該 dataset"
        )

os.chdir(WORK_DIR)
print(f"📂 Working directory: {os.getcwd()}")
print(f"   Files: {os.listdir('.')[:10]}")

In [ ]:
# 安裝依賴（用 pip，不依賴 uv）
# autoresearch 的 pyproject.toml 定義了依賴，嘗試用 pip 安裝
import subprocess, sys

# 嘗試 pip install from pyproject.toml
if os.path.exists("pyproject.toml"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=False)
    print("✅ Dependencies installed via pip")
else:
    print("⚠️ No pyproject.toml found, assuming dependencies are pre-installed")

In [ ]:
# T4/P100 相容性 patch（內嵌版，不需要下載）
import re, torch
from pathlib import Path

def get_gpu_capability():
    try:
        if torch.cuda.is_available():
            cap = torch.cuda.get_device_capability(0)
            return cap[0] * 10 + cap[1]
    except Exception:
        pass
    return 0

def patch_train_py(train_path):
    text = train_path.read_text()
    original = text
    is_ampere = get_gpu_capability() >= 80

    # Patch 1: Flash Attention 3 → SDPA fallback
    fa3_replacement = """import torch
_gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
_is_ampere_plus = _gpu_cap[0] >= 8

if _is_ampere_plus:
    import kernels
    fa3 = kernels.get_kernel("flash_attn3")
else:
    fa3 = None  # Will use SDPA fallback"""

    if 'import kernels' in text and 'fa3 = kernels.get_kernel' in text:
        text = re.sub(
            r'import kernels\s*\nfa3\s*=\s*kernels\.get_kernel\(["\']flash_attn3["\']\)',
            fa3_replacement, text,
        )

    # Patch 2: Replace fa3 call with fallback
    sdpa_fallback = """if fa3 is not None:
            y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
        else:
            # SDPA fallback for T4 / non-Ampere GPUs
            y = torch.nn.functional.scaled_dot_product_attention(
                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
                is_causal=True,
            ).transpose(1, 2)"""
    text = re.sub(
        r'y = fa3\.flash_attn_func\(q, k, v, causal=True, window_size=window_size\)',
        sdpa_fallback, text,
    )

    # Patch 3: bfloat16 → float16 for T4
    if not is_ampere:
        text = text.replace('torch.bfloat16', 'torch.float16')
        text = re.sub(
            r"autocast\(device_type=['\"]cuda['\"],\s*dtype=torch\.bfloat16\)",
            "autocast(device_type='cuda', dtype=torch.float16)", text,
        )

    if text == original:
        print("[t4_patch] No changes needed (already patched or Ampere+ GPU)")
        return False
    train_path.write_text(text)
    return True

# 套用 patch
train_path = Path("train.py")
cap = get_gpu_capability()
print(f"[t4_patch] GPU compute capability: {cap}")

if cap >= 80:
    print("[t4_patch] Ampere+ GPU — no patch needed")
else:
    print(f"[t4_patch] Pre-Ampere GPU (sm_{cap}) — applying T4 compatibility patch...")
    if patch_train_py(train_path):
        print("[t4_patch] Done! Patched:")
        print("  - Flash Attention 3 → SDPA fallback")
        if cap < 80:
            print("  - bfloat16 → float16")

In [ ]:
# 資料準備（下載 + tokenizer，約 2-3 分鐘）
# 注意：prepare.py 需要下載資料集，如果無網路會失敗
# 可以預先將資料也加入 Kaggle dataset
!python prepare.py --num-shards 10

In [ ]:
# 跑一次訓練（約 5-10 分鐘，視 GPU 而定）
!python train.py 2>&1 | tee /tmp/run.log

# 提取結果
import re
log = open("/tmp/run.log").read()
bpb_match = re.findall(r"val_bpb[:\s]+([\d.]+)", log)
loss_match = re.findall(r"train_loss[:\s]+([\d.]+)", log)
step_match = re.findall(r"step[:\s]+(\d+)", log)

val_bpb = float(bpb_match[-1]) if bpb_match else None
train_loss = float(loss_match[-1]) if loss_match else None
steps = int(step_match[-1]) if step_match else None

print(f"\n{'='*40}")
print(f"val_bpb:    {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps:      {steps}")
print(f"{'='*40}")

## 回傳結果到 MD.Piece API

設定你的 MD.Piece 後端 URL。

**方式 1：** 用 ngrok 暴露本機 port 8000

**方式 2：** 部署到公開伺服器

**方式 3：** 留空，結果存在本地 TSV，之後手動上傳

In [ ]:
# ===== 設定區 =====
API_URL = ""  # 填入你的 MD.Piece 後端 URL，例如 "https://xxxx.ngrok.io"
EXPERIMENT_NAME = "baseline-kaggle"
NOTES = "Kaggle GPU baseline run"
# ==================

import requests, json

payload = {
    "name": EXPERIMENT_NAME,
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "notes": NOTES,
    "kept": True,
}

if API_URL:
    try:
        r = requests.post(f"{API_URL}/research/", json=payload, timeout=10)
        if r.ok:
            print(f"✅ 已回傳到 MD.Piece: {r.json()}")
        else:
            print(f"⚠️ API 回傳錯誤 ({r.status_code}): {r.text}")
    except Exception as e:
        print(f"⚠️ 無法連線到 {API_URL}: {e}")
        print(f"📋 手動提交指令：")
        print(f'curl -X POST {API_URL}/research/ -H "Content-Type: application/json" -d \'{json.dumps(payload)}\'')
else:
    print("ℹ️ 未設定 API_URL，結果僅存在本地")
    print(f"📋 結果：{json.dumps(payload, indent=2)}")

## 自動化實驗循環

內嵌 `autoresearch_runner.py`，自動跑多輪實驗（假設 → 修改 → 訓練 → 評估 → keep/revert）。不需要下載任何檔案。

In [ ]:
# 內嵌 autoresearch_runner.py（不需要網路下載）
import argparse, json, os, re, subprocess, sys, time
from datetime import datetime
from pathlib import Path

# ── 實驗假設列表 ──
HYPOTHESES = [
    {"name": "deeper-model", "description": "增加模型深度 n_layer 8→12",
     "patch": {"target": "n_layer", "old": "8", "new": "12"}},
    {"name": "wider-model", "description": "增加模型寬度 n_embd 512→768",
     "patch": {"target": "n_embd", "old": "512", "new": "768"}},
    {"name": "more-heads", "description": "增加注意力頭數 n_head 4→8",
     "patch": {"target": "n_head", "old": "4", "new": "8"}},
    {"name": "larger-batch", "description": "增大 batch size ×2",
     "patch": {"target": "TOTAL_BATCH_SIZE", "find_multiply": 2}},
    {"name": "higher-lr", "description": "提高學習率 ×1.5",
     "patch": {"target": "learning_rate", "find_multiply": 1.5}},
    {"name": "lower-lr", "description": "降低學習率 ×0.5",
     "patch": {"target": "learning_rate", "find_multiply": 0.5}},
    {"name": "longer-warmup", "description": "增加 warmup 步數 ×2",
     "patch": {"target": "warmup_steps", "find_multiply": 2}},
    {"name": "window-pattern-SSSS", "description": "attention window 全 sliding",
     "patch": {"target": "window_pattern", "old": "'SSSL'", "new": "'SSSS'"}},
    {"name": "window-pattern-SLSL", "description": "attention window 交替",
     "patch": {"target": "window_pattern", "old": "'SSSL'", "new": "'SLSL'"}},
    {"name": "sequence-len-4096", "description": "序列長度 2048→4096",
     "patch": {"target": "sequence_len", "old": "2048", "new": "4096"}},
]

TSV_HEADER = "name\tval_bpb\ttrain_loss\tsteps\tduration\tkept\tdescription\ttimestamp\n"

def init_results_tsv(path):
    if not path.exists():
        path.write_text(TSV_HEADER)

def append_result(path, result):
    line = (f"{result['name']}\t{result.get('val_bpb', '')}\t{result.get('train_loss', '')}\t"
            f"{result.get('steps', '')}\t{result.get('duration_seconds', '')}\t"
            f"{result.get('kept', '')}\t{result.get('description', '')}\t"
            f"{result.get('timestamp', '')}\n")
    with open(path, "a") as f:
        f.write(line)

def run_training(timeout_extra=60):
    print("  ⏳ Training...")
    start = time.time()
    try:
        result = subprocess.run(
            [sys.executable, "train.py"],
            capture_output=True, text=True, timeout=600 + timeout_extra,
        )
        log = result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        return {"error": "Training timed out", "duration_seconds": time.time() - start}
    duration = time.time() - start
    bpb_matches = re.findall(r"val(?:_|\s)bpb[:\s]+([\d.]+)", log)
    loss_matches = re.findall(r"train(?:_|\s)loss[:\s]+([\d.]+)", log)
    step_matches = re.findall(r"step[:\s]+(\d+)", log)
    val_bpb = float(bpb_matches[-1]) if bpb_matches else None
    train_loss = float(loss_matches[-1]) if loss_matches else None
    steps = int(step_matches[-1]) if step_matches else None
    if val_bpb is None and result.returncode != 0:
        error_lines = [l for l in log.split("\n") if "Error" in l or "error" in l]
        return {"error": error_lines[-1] if error_lines else "Training failed", "duration_seconds": duration}
    return {"val_bpb": val_bpb, "train_loss": train_loss, "steps": steps, "duration_seconds": round(duration, 1)}

def git_commit(message):
    subprocess.run(["git", "add", "-A"], check=True)
    subprocess.run(["git", "commit", "-am", message], check=True)

def git_revert():
    subprocess.run(["git", "reset", "--hard", "HEAD~1"], check=True)

def apply_patch(train_path, patch):
    text = train_path.read_text()
    target = patch["target"]
    if "old" in patch and "new" in patch:
        if patch["old"] not in text:
            print(f"  ⚠️ Could not find '{patch['old']}' in train.py, skipping")
            return False
        text = text.replace(patch["old"], patch["new"], 1)
    elif "find_multiply" in patch:
        pattern = rf"({target}\s*=\s*)([\d.]+)"
        match = re.search(pattern, text)
        if not match:
            print(f"  ⚠️ Could not find '{target} = <number>' in train.py, skipping")
            return False
        old_val = float(match.group(2))
        new_val = old_val * patch["find_multiply"]
        if new_val == int(new_val):
            new_val = int(new_val)
        text = text[:match.start()] + f"{match.group(1)}{new_val}" + text[match.end():]
    else:
        return False
    train_path.write_text(text)
    return True

def submit_to_api(api_url, result):
    if not api_url:
        return
    try:
        import requests
        payload = {"name": result["name"], "val_bpb": result.get("val_bpb"),
                   "train_loss": result.get("train_loss"), "steps": result.get("steps"),
                   "duration_seconds": result.get("duration_seconds"),
                   "notes": result.get("description", ""), "kept": result.get("kept", False)}
        r = requests.post(f"{api_url}/research/", json=payload, timeout=10)
        if r.ok:
            print(f"  📡 Submitted to API: {r.json().get('message')}")
        else:
            print(f"  ⚠️ API error {r.status_code}: {r.text[:100]}")
    except Exception as e:
        print(f"  ⚠️ Could not reach API: {e}")

# ===== 設定區 =====
API_URL = ""  # 填入你的 MD.Piece 後端 URL（留空則不回傳）
ROUNDS = 5
# ==================

train_path = Path("train.py")
if not train_path.exists():
    raise FileNotFoundError("❌ train.py not found. 請確認 autoresearch dataset 已正確載入")

results_path = Path("results.tsv")
init_results_tsv(results_path)

# 初始化 git（runner 需要 git commit/revert）
!git config user.email "kaggle@autoresearch" && git config user.name "autoresearch"
!git init . 2>/dev/null; git add -A; git commit -m "initial" --allow-empty 2>/dev/null

# ── Baseline ──
print("=" * 60)
print(f"🔬 AutoResearch Runner (Rounds: {ROUNDS})")
print("=" * 60)
print("\n📊 Round 0: Baseline")
baseline = run_training()

if "error" in baseline:
    print(f"  ❌ Baseline failed: {baseline['error']}")
    raise RuntimeError("Baseline training failed")

best_bpb = baseline["val_bpb"]
print(f"  ✅ Baseline val_bpb: {best_bpb}")

baseline_result = {"name": "baseline", "description": "Initial baseline run",
                   "kept": True, "timestamp": datetime.utcnow().isoformat(), **baseline}
append_result(results_path, baseline_result)
submit_to_api(API_URL, baseline_result)

# ── Experiment loop ──
hypotheses = HYPOTHESES[:ROUNDS]
for i, hyp in enumerate(hypotheses, 1):
    print(f"\n{'='*60}")
    print(f"🧪 Round {i}/{len(hypotheses)}: {hyp['name']}")
    print(f"   Hypothesis: {hyp['description']}")

    if not apply_patch(train_path, hyp["patch"]):
        print("  ⏭️ Skipping (patch failed)")
        continue

    git_commit(f"hypothesis: {hyp['description']}")
    result = run_training()

    if "error" in result:
        print(f"  ❌ Training failed: {result['error']}")
        kept = False
        git_revert()
    elif result["val_bpb"] is not None and result["val_bpb"] < best_bpb:
        improvement = best_bpb - result["val_bpb"]
        print(f"  ✅ KEPT — val_bpb: {result['val_bpb']} (improved by {improvement:.4f})")
        best_bpb = result["val_bpb"]
        kept = True
    else:
        val = result.get("val_bpb", "N/A")
        print(f"  ↩️ REVERTED — val_bpb: {val} (no improvement over {best_bpb})")
        git_revert()
        kept = False

    exp_result = {"name": hyp["name"], "description": hyp["description"],
                  "kept": kept, "timestamp": datetime.utcnow().isoformat(),
                  **{k: v for k, v in result.items() if k != "error"}}
    append_result(results_path, exp_result)
    submit_to_api(API_URL, exp_result)

print(f"\n{'='*60}")
print(f"🏁 Done! Best val_bpb: {best_bpb}")
print(f"   Results saved to: {results_path}")
print(f"{'='*60}")

In [ ]:
# 查看結果摘要
import pandas as pd
from pathlib import Path

results_file = Path("results.tsv")
if not results_file.exists():
    # 可能在 autoresearch/ 子目錄
    alt = Path("autoresearch/results.tsv")
    if alt.exists():
        results_file = alt

if results_file.exists() and results_file.stat().st_size > 0:
    df = pd.read_csv(results_file, sep="\t")
    print(df.to_string(index=False))
    if "val_bpb" in df.columns and df["val_bpb"].notna().any():
        best_idx = df["val_bpb"].idxmin()
        print(f"\n🏆 Best: {df.loc[best_idx, 'name']} — val_bpb: {df['val_bpb'].min()}")
    else:
        print("\n⚠️ No valid val_bpb values found")
else:
    print("❌ results.tsv not found — the runner may have failed.")
    print("   Check the output of the previous cell for errors.")
    print("   Common causes:")
    print("   1. autoresearch_runner.py download failed")
    print("   2. Baseline training failed (GPU not enabled?)")
    print("   3. train.py not found (git clone failed)")

In [ ]:
# 下載 results.tsv（Kaggle 可直接從 Output 下載）
from pathlib import Path
from IPython.display import FileLink, display

for p in [Path("results.tsv"), Path("autoresearch/results.tsv")]:
    if p.exists():
        display(FileLink(str(p)))
        break
else:
    print("⚠️ results.tsv not found — nothing to download")